# Solutions · 12 · External mass transfer and packed-bed reactors

Worked solutions to the exercises of [notebook 12](../notebooks/12_external_mass_transfer_and_packed_beds.ipynb). Try each exercise
yourself before reading its solution - the learning happens in the attempt. Solutions are one way to
answer each question; other correct approaches exist.

In [1]:
try:                      # on Google Colab (or anywhere engreact is missing): install it from GitHub
    import engreact
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "git+https://github.com/ktwyw/engreact"], check=True)
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from engreact import catalysis, ideal
plt.rcParams.update({"figure.figsize": (7, 4), "figure.dpi": 90, "axes.grid": True, "grid.alpha": 0.3,
                     "axes.spines.top": False, "axes.spines.right": False})

import inspect
from IPython.display import Code

def show_source(obj):
    """Display the source code of a library function or class."""
    return Code(inspect.getsource(obj), language="python")

def heading(text):
    print(f"\n{text}\n" + "=" * len(text))

## Exercise 1

A catalytic converter oxidises CO at 800 K on a monolith (k_c a_c = 60 1/s at U = 5 m/s). Find the monolith length for 99 % conversion, and the apparent activation energy you would measure if the film controls (D_AB ~ T^1.75).

In [2]:
kc_ac, U = 60.0, 5.0                                             # 1/s, m/s
L99 = -np.log(0.01) * U / kc_ac
print(f"monolith length for 99 % CO conversion: {L99:.2f} m")
for T1, T2 in ((700.0, 800.0),):
    ratio = (T2 / T1) ** 1.75 * (T1 / T2)                             # D_AB ~ T^1.75, concentration ~ 1/T at fixed pressure... k_c a_c ~ D_AB^(2/3) roughly
    Ea_app = 8.314 * np.log((T2 / T1) ** (1.75 * 2 / 3)) / (1 / T1 - 1 / T2)
    print(f"apparent activation energy in the film-controlled regime: about {Ea_app/1e3:.1f} kJ/mol (from k_c ~ D_AB^(2/3) ~ T^1.17)")

monolith length for 99 % CO conversion: 0.38 m
apparent activation energy in the film-controlled regime: about 7.3 kJ/mol (from k_c ~ D_AB^(2/3) ~ T^1.17)


A film-controlled converter needs a fraction of a metre, and its "activation energy" is the temperature
dependence of gas diffusion - a few kJ/mol against the 50-100 of surface kinetics. An Arrhenius plot that
flattens to that slope at high temperature is the signature of external mass-transfer control.

## Exercise 2

For the packed bed of the notebook, compute the pressure drop (Ergun, notebook 04) for the bed lengths needed at U = 0.2, 0.5 and 1.0 m/s. Which velocity minimises the compressor work per unit conversion?

In [3]:
D_AB, nu, d_p, a_c, rho, mu, phi = 1.5e-5, 1.6e-5, 3e-3, 1200.0, 0.9, 2.5e-5, 0.4
Sc = nu / D_AB
for U in (0.2, 0.5, 1.0):
    k_c = catalysis.sherwood_frossling(U * d_p / nu, Sc) * D_AB / d_p
    L99 = -np.log(0.01) * U / (k_c * a_c)
    dP = ideal.ergun(rho * U, rho, mu, d_p, phi) * L99
    print(f"U = {U} m/s: L = {L99:.3f} m, pressure drop {dP:.0f} Pa, compressor work per unit gas ~ dP: {dP:.0f} J/m3")

U = 0.2 m/s: L = 0.027 m, pressure drop 18 Pa, compressor work per unit gas ~ dP: 18 J/m3
U = 0.5 m/s: L = 0.048 m, pressure drop 116 Pa, compressor work per unit gas ~ dP: 116 J/m3
U = 1.0 m/s: L = 0.074 m, pressure drop 536 Pa, compressor work per unit gas ~ dP: 536 J/m3


Both the bed length (as $U^{1/2}$) and the Ergun gradient (as $U$ to $U^2$) grow with velocity, so the pressure
drop for a fixed conversion rises steeply: the lowest velocity that still gives acceptable throughput
minimises compressor work. Mass-transfer-limited beds are run slow and wide.

## Exercise 3

**Implement it yourself:** write the Frossling correlation and compute k_c for pellets of 1, 3 and 10 mm at U = 0.5 m/s; explain why small pellets have larger k_c but also a larger pressure drop.

In [4]:
def frossling(Re, Sc):
    return 2 + 0.6 * Re ** 0.5 * Sc ** (1 / 3)
for d in (1e-3, 3e-3, 10e-3):
    Re = 0.5 * d / nu
    k_c = frossling(Re, Sc) * D_AB / d
    print(f"d_p = {d*1e3:4.0f} mm: Sh = {frossling(Re, Sc):.1f}, k_c = {k_c*1e3:.1f} mm/s (library {catalysis.sherwood_frossling(Re, Sc)*D_AB/d*1e3:.1f}); Ergun gradient {ideal.ergun(rho*0.5, rho, mu, d, phi):.0f} Pa/m")

d_p =    1 mm: Sh = 5.4, k_c = 81.4 mm/s (library 81.4); Ergun gradient 14238 Pa/m
d_p =    3 mm: Sh = 7.9, k_c = 39.7 mm/s (library 39.7); Ergun gradient 2402 Pa/m
d_p =   10 mm: Sh = 12.8, k_c = 19.3 mm/s (library 19.3); Ergun gradient 475 Pa/m


Small pellets have thin films and large mass-transfer coefficients (Sh grows slower than $d_p$, so $k_c
\propto d_p^{-1/2}$ or so) and a large interfacial area, but the pressure gradient grows even faster; the
same trade-off as internal diffusion, with the same resolution in practice - a few millimetres.